# 🏥 Day-of Surgery Cancellation Analysis

## Project Overview
This project analyzes same-day surgery cancellations to identify leading
causes, high-risk specialties, and scheduling patterns that contribute to
cancellations — helping OR scheduling teams target interventions that
reduce wasted OR time and improve patient outcomes.

## Stakeholder Questions
1. What are the leading reasons for same-day cancellations?
2. Which surgical specialties have the highest cancellation rates?
3. Do cases scheduled later in the day have higher cancellation rates?
4. Is there a relationship between patient age, ASA status, and
   cancellation reason?
5. What is the estimated cost impact of cancellations by reason?

## Tools Used
- Python (pandas, numpy) — data generation and manipulation
- SQLite — SQL analysis (JOIN, HAVING, CASE WHEN, RANK window function)
- Tableau — operational dashboard

## Data
Synthetic datasets generated to simulate real OR scheduling records,
with cancellation reason distributions informed by published clinical
research on day-of surgery cancellations.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# ── DATASET 1: SCHEDULED SURGERIES ───────────────────────────────────────────
num_surgeries = 1200

specialties = ['Orthopedic', 'General Surgery', 'Vascular',
                'Pulmonary', 'Ophthalmic', 'Gynecologic']

# Specialty distribution roughly matching research (Ortho highest volume/risk)
specialty_weights = [30, 25, 12, 10, 13, 10]
specialty_probs = [w/sum(specialty_weights) for w in specialty_weights]

# Scheduled times across the OR day (7am - 5pm), weighted toward morning
hour_weights = [10,10,9,8,7,6,5,5,5,4,4,4]  # hours 7-18
hour_probs = [w/sum(hour_weights) for w in hour_weights]
hours_range = list(range(7, 19))

scheduled_hours = np.random.choice(hours_range, num_surgeries, p=hour_probs)
scheduled_minutes = np.random.choice([0, 15, 30, 45], num_surgeries)

# Day of week (Mon-Fri only, OR scheduling)
days = np.random.choice(['Mon','Tue','Wed','Thu','Fri'], num_surgeries,
                          p=[0.22, 0.22, 0.20, 0.18, 0.18])

scheduled_surgeries_df = pd.DataFrame({
    'surgery_id'        : range(1, num_surgeries + 1),
    'patient_id'        : range(1, num_surgeries + 1),
    'age'               : np.random.randint(2, 95, num_surgeries),
    'specialty'         : np.random.choice(specialties, num_surgeries, p=specialty_probs),
    'scheduled_hour'    : scheduled_hours,
    'scheduled_minute'  : scheduled_minutes,
    'day_of_week'       : days,
    'interpreter_needed': np.random.choice(['Y','N'], num_surgeries, p=[0.12, 0.88]),
    'asa_status'        : np.random.choice([1,2,3,4], num_surgeries, p=[0.20, 0.40, 0.30, 0.10])
})

# ── DATASET 2: CANCELLATIONS ──────────────────────────────────────────────────
# Overall cancellation rate ~12% (realistic for OR settings)
cancelled_flags = np.random.choice(['Y','N'], num_surgeries, p=[0.12, 0.88])

# Cancellation reason distribution based on published research
cancellation_reasons = [
    'Lack of OR Time', 'Patient No-Show', 'Medical Reasons (NPO/Meds/BP)',
    'Change in Surgical Plan', 'Administrative', 'Miscellaneous'
]
reason_probs = [0.597, 0.162, 0.108, 0.054, 0.037, 0.042]

reasons = []
cost_impacts = []
for flag in cancelled_flags:
    if flag == 'Y':
        reason = np.random.choice(cancellation_reasons, p=reason_probs)
        reasons.append(reason)
        # Cost impact varies by reason - OR time is expensive
        if reason == 'Lack of OR Time':
            cost = np.random.randint(2000, 6000)
        elif reason == 'Patient No-Show':
            cost = np.random.randint(1500, 4000)
        elif reason == 'Medical Reasons (NPO/Meds/BP)':
            cost = np.random.randint(1500, 5000)
        else:
            cost = np.random.randint(1000, 3500)
        cost_impacts.append(cost)
    else:
        reasons.append(None)
        cost_impacts.append(0)

cancellations_df = pd.DataFrame({
    'surgery_id'             : range(1, num_surgeries + 1),
    'cancelled'              : cancelled_flags,
    'cancellation_reason'    : reasons,
    'estimated_cost_impact'  : cost_impacts
})

print("✅ Scheduled Surgeries:", scheduled_surgeries_df.shape)
print("✅ Cancellations:", cancellations_df.shape)
print("\n--- Scheduled Surgeries Sample ---")
print(scheduled_surgeries_df.head())
print("\n--- Cancellations Sample ---")
print(cancellations_df.head())
print("\nCancellation rate:", (cancelled_flags == 'Y').mean())

✅ Scheduled Surgeries: (1200, 9)
✅ Cancellations: (1200, 4)

--- Scheduled Surgeries Sample ---
   surgery_id  patient_id  age        specialty  scheduled_hour  \
0           1           1   58       Orthopedic               9   
1           2           2   73         Vascular              18   
2           3           3    8       Orthopedic              14   
3           4           4   79       Ophthalmic              12   
4           5           5   57  General Surgery               8   

   scheduled_minute day_of_week interpreter_needed  asa_status  
0                 0         Fri                  N           3  
1                45         Thu                  N           2  
2                30         Tue                  N           1  
3                 0         Tue                  Y           1  
4                30         Tue                  N           4  

--- Cancellations Sample ---
   surgery_id cancelled cancellation_reason  estimated_cost_impact
0           1 

In [2]:
conn = sqlite3.connect(':memory:')

scheduled_surgeries_df.to_sql('scheduled_surgeries', conn, index=False, if_exists='replace')
cancellations_df.to_sql('cancellations', conn, index=False, if_exists='replace')

print("✅ Both tables loaded into SQLite successfully!")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

✅ Both tables loaded into SQLite successfully!
                  name
0  scheduled_surgeries
1        cancellations


## Query 1 — Leading Reasons for Same-Day Cancellations
**Stakeholder Question 1:** What are the leading reasons for same-day cancellations?

Lack of OR Time accounts for 63.3% of cancellations — a scheduling
and block management issue. Patient No-Show follows at 16.3%,
suggesting a need for stronger reminder and communication protocols.
Medical reasons (NPO violations, uncontrolled BP, medications) at 8.2%
point to opportunities in pre-operative patient education.

In [3]:
query1 = """
SELECT
    c.cancellation_reason,
    COUNT(*)                              AS total_cancellations,
    ROUND(COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM cancellations
         WHERE cancelled = 'Y'), 1)       AS pct_of_cancellations
FROM cancellations c
WHERE c.cancelled = 'Y'
GROUP BY c.cancellation_reason
ORDER BY total_cancellations DESC;
"""

top_cancellation_reasons = pd.read_sql(query1, conn)
print(top_cancellation_reasons)

             cancellation_reason  total_cancellations  pct_of_cancellations
0                Lack of OR Time                   93                  63.3
1                Patient No-Show                   24                  16.3
2  Medical Reasons (NPO/Meds/BP)                   12                   8.2
3                  Miscellaneous                    8                   5.4
4        Change in Surgical Plan                    7                   4.8
5                 Administrative                    3                   2.0


In [4]:
query2 = """
SELECT
    s.specialty,
    COUNT(*)                                AS total_cancellations,
    ROUND(COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM scheduled_surgeries
         WHERE specialty = s.specialty), 1) AS cancellation_rate_pct
FROM scheduled_surgeries s
JOIN cancellations c ON s.surgery_id = c.surgery_id
WHERE c.cancelled = 'Y'
GROUP BY s.specialty
ORDER BY total_cancellations DESC;
"""

cancellations_by_specialty = pd.read_sql(query2, conn)
print(cancellations_by_specialty)

         specialty  total_cancellations  cancellation_rate_pct
0       Orthopedic                   48                   12.8
1  General Surgery                   32                   10.6
2      Gynecologic                   23                   22.1
3         Vascular                   18                   12.4
4       Ophthalmic                   14                    8.8
5        Pulmonary                   12                   10.3


## Query 3 — Cancellation Rate by Time of Day
**Stakeholder Question 3:** Do cases scheduled later in the day have
higher cancellation rates?

Rates are consistent across the day: Morning 11.1%, Midday 12.5%,
Afternoon 13.9%. The narrow spread suggests time of day is not a
primary cancellation driver. The slight afternoon increase is consistent
with OR time squeeze patterns when earlier cases run long.

In [5]:
query3 = """
SELECT
    CASE
        WHEN s.scheduled_hour BETWEEN 7 AND 10  THEN 'Morning'
        WHEN s.scheduled_hour BETWEEN 11 AND 13 THEN 'Midday'
        WHEN s.scheduled_hour BETWEEN 14 AND 18 THEN 'Afternoon'
    END                                              AS time_of_day,
    COUNT(*)                                         AS total_surgeries,
    SUM(CASE WHEN c.cancelled = 'Y' THEN 1
             ELSE 0 END)                             AS total_cancellations,
    ROUND(SUM(CASE WHEN c.cancelled = 'Y' THEN 1
             ELSE 0 END) * 100.0 / COUNT(*), 1)     AS cancellation_rate_pct
FROM scheduled_surgeries s
JOIN cancellations c ON s.surgery_id = c.surgery_id
GROUP BY time_of_day
ORDER BY cancellation_rate_pct DESC;
"""

cancellations_by_time = pd.read_sql(query3, conn)
print(cancellations_by_time)

  time_of_day  total_surgeries  total_cancellations  cancellation_rate_pct
0   Afternoon              346                   48                   13.9
1      Midday              287                   36                   12.5
2     Morning              567                   63                   11.1


## Query 4 — Medical Cancellations by Age Group & ASA Status
**Stakeholder Question 4:** Is there a relationship between patient age,
ASA status and cancellation reason?

Medical cancellations are evenly distributed across age groups with no
single group at significantly higher risk. Average ASA status of 2.0-3.0
suggests patients with pre-existing conditions need stronger pre-operative
screening and education protocols regardless of age.

In [6]:
query4 = """
SELECT
    CASE
        WHEN s.age BETWEEN 0  AND 17 THEN 'Under 18'
        WHEN s.age BETWEEN 18 AND 44 THEN '18-44'
        WHEN s.age BETWEEN 45 AND 64 THEN '45-64'
        WHEN s.age BETWEEN 65 AND 79 THEN '65-79'
        ELSE '80+'
    END                          AS age_group,
    s.asa_status,
    c.cancellation_reason,
    COUNT(*)                     AS total_cancellations
FROM scheduled_surgeries s
JOIN cancellations c ON s.surgery_id = c.surgery_id
WHERE c.cancelled = 'Y'
AND c.cancellation_reason IN (
    'Lack of OR Time',
    'Patient No-Show',
    'Medical Reasons (NPO/Meds/BP)'
)
GROUP BY age_group, s.asa_status, c.cancellation_reason
ORDER BY age_group, total_cancellations DESC;
"""

cancellations_by_age = pd.read_sql(query4, conn)
print(cancellations_by_age)

   age_group  asa_status            cancellation_reason  total_cancellations
0      18-44           2                Lack of OR Time                   10
1      18-44           1                Lack of OR Time                    6
2      18-44           3                Lack of OR Time                    6
3      18-44           2                Patient No-Show                    3
4      18-44           3                Patient No-Show                    2
5      18-44           4                Lack of OR Time                    2
6      18-44           2  Medical Reasons (NPO/Meds/BP)                    1
7      18-44           3  Medical Reasons (NPO/Meds/BP)                    1
8      18-44           4  Medical Reasons (NPO/Meds/BP)                    1
9      45-64           2                Lack of OR Time                    7
10     45-64           3                Lack of OR Time                    5
11     45-64           1                Lack of OR Time                    4

In [7]:
query4b = """
SELECT
    CASE
        WHEN s.age BETWEEN 0  AND 17 THEN 'Under 18'
        WHEN s.age BETWEEN 18 AND 44 THEN '18-44'
        WHEN s.age BETWEEN 45 AND 64 THEN '45-64'
        WHEN s.age BETWEEN 65 AND 79 THEN '65-79'
        ELSE '80+'
    END                      AS age_group,
    ROUND(AVG(s.asa_status), 1) AS avg_asa_status,
    COUNT(*)                 AS total_cancellations
FROM scheduled_surgeries s
JOIN cancellations c ON s.surgery_id = c.surgery_id
WHERE c.cancelled = 'Y'
AND c.cancellation_reason = 'Medical Reasons (NPO/Meds/BP)'
GROUP BY age_group
ORDER BY total_cancellations DESC;
"""

medical_by_age = pd.read_sql(query4b, conn)
print(medical_by_age)

  age_group  avg_asa_status  total_cancellations
0  Under 18             2.7                    3
1     45-64             2.3                    3
2     18-44             3.0                    3
3       80+             2.5                    2
4     65-79             2.0                    1


## Query 5 — Estimated Cost Impact by Cancellation Reason
**Stakeholder Question 5:** What is the estimated cost impact of
cancellations by reason?

Lack of OR Time drives $362,989 in estimated costs — nearly 5x the
next highest reason. Patient No-Shows add $74,213. Together these two
reasons represent over $437,000 in potentially recoverable revenue
through improved scheduling practices and patient communication protocols.

In [8]:
query5 = """
SELECT
    c.cancellation_reason,
    COUNT(*)                              AS total_cancellations,
    SUM(c.estimated_cost_impact)          AS total_cost_impact,
    ROUND(AVG(c.estimated_cost_impact),0) AS avg_cost_per_cancellation
FROM cancellations c
WHERE c.cancelled = 'Y'
GROUP BY c.cancellation_reason
ORDER BY total_cost_impact DESC;
"""

cost_by_reason = pd.read_sql(query5, conn)
print(cost_by_reason)

             cancellation_reason  total_cancellations  total_cost_impact  \
0                Lack of OR Time                   93             362989   
1                Patient No-Show                   24              74213   
2  Medical Reasons (NPO/Meds/BP)                   12              36892   
3                  Miscellaneous                    8              19354   
4        Change in Surgical Plan                    7              15195   
5                 Administrative                    3               6033   

   avg_cost_per_cancellation  
0                     3903.0  
1                     3092.0  
2                     3074.0  
3                     2419.0  
4                     2171.0  
5                     2011.0  
